# 01 · CDI mapping by WordNet hyponym pooling

**What it does.** Maps the dataset's classes (ImageNet's 1,000 classes or LVIS's 1,203 categories) onto the 295 CDI
nouns. A CDI word pools **every class whose WordNet synset is one of the word's anchor synsets or a hyponym of
one**, minus classes excluded by hand.

**Decisions**
- *Sense choice.* WordNet's first sense is often wrong for child words (arm → weapon, toy → toy dog), so every
  mapped word lists its anchors explicitly.
- *Added anchors* where WordNet splits one everyday object (cup + coffee mug), *exclusions* where a hyponym is not
  the referent (punching bag from ball).
- *LVIS* starts from the ImageNet anchors and layers LVIS-specific changes on top, so a CDI word means the same
  thing in both datasets and the ImageNet mapping is unaffected.
- A class can pool into several CDI words (hen → hen, chicken, bird, animal); `most_specific` marks the closest.
  The analyses use the *exclusive* sets built in notebook 05, where each class counts once.
- *LVIS only:* this step also writes the **crop manifest** (one row per labeled box) for embedding. No CLIP filter,
  no size filter; zero-area boxes are dropped.

**Outputs** (`outputs/mapping/`): `{imagenet,lvis}_cdi_hyponym_map.csv`, `cdi_{imagenet,lvis}_summary.csv`,
`imagenet_class_to_cdi.csv` or `lvis_crop_manifest.csv` + `lvis_crop_metadata.csv.gz`, and
`anchors_{imagenet,lvis}.json` (read by notebook 02).

In [1]:
# Standard setup shared by every notebook (each notebook is self-contained).
import json
import os
import random
import re
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch, Rectangle
from scipy.cluster.hierarchy import leaves_list, linkage, optimal_leaf_ordering
from scipy.spatial.distance import pdist, squareform
from scipy.stats import false_discovery_control, pearsonr, rankdata, spearmanr, wilcoxon
from IPython.display import Image, Markdown, display

plt.rcParams["font.family"] = ["DejaVu Sans", "sans-serif"]
pd.set_option("display.max_colwidth", 120)


def show_images(paths, width=950):
    """Display saved figures (the files this notebook writes)."""
    for p in map(Path, paths):
        if p.exists():
            display(Markdown(f"`{p.relative_to(ROOT)}`")); display(Image(filename=str(p), width=width))
        else:
            display(Markdown(f"*Missing figure:* `{p}`"))

In [2]:
# Paths. Run this notebook from analysis/comparison-datasets/notebooks/ (Jupyter's default working directory).
ROOT = Path.cwd().resolve().parent                     # analysis/comparison-datasets
assert (ROOT / "notebooks").is_dir(), "open this notebook from analysis/comparison-datasets/notebooks/"
REPO = ROOT.parents[1]
DATA_DIR = REPO / "data"
MANUSCRIPT_DIR = REPO / "analysis" / "manuscript-2026"
VEDB_DIR = REPO / "analysis" / "vedb-1612"
VOCAB_DIR = DATA_DIR / "vocabulary_lists"
VALID129_TXT = VOCAB_DIR / "included_categories_valid129.txt"

# Inputs written by other pipelines (read only).
EXEMPLAR_EMBED_ROOT = Path(os.environ.get("BV_EXEMPLAR_EMBED_ROOT",
                                          str(MANUSCRIPT_DIR / "exemplar_set_embeddings"))).expanduser()
BV_THINGS_VALID129 = EXEMPLAR_EMBED_ROOT / "valid129"       # manuscript Stage 0: BV + THINGS tables
VEDB_VALID129 = VEDB_DIR / "exemplar_set_embeddings" / "valid129"   # vedb-1612 Stage 0 (files use the "bv_" prefix)

# External data on ccn2.
IMAGENET_DIR = Path("/data2/j7yang/imagenet-1k/extracted")
IMAGENET_EMBED_ROOT = Path("/ccn2a/dataset/babyview/related_data/imagenet_1k/outputs/image_embeddings")
LVIS_ROOT = Path("/data2/j7yang/lvis")
LVIS_CROP_EMBED_ROOT = Path("/ccn2a/dataset/babyview/related_data/lvis/outputs/crop_embeddings")

# Outputs of this folder.
OUT = ROOT / "outputs"
MAPPING_OUT = OUT / "mapping"
MONTAGE_OUT = OUT / "montages"
EXEMPLAR_SETS = OUT / "exemplar_sets"
MODELS = ("clip", "dinov3")

## Parameters
`DATASET` = `"imagenet"` or `"lvis"`.

In [3]:
DATASET = "imagenet"
assert DATASET in ('imagenet', 'lvis'), "DATASET must be one of ('imagenet', 'lvis')"

## ImageNet anchors
Curated WordNet anchors for mapping the 295 CDI nouns onto ImageNet-1k classes.

Hyponym pooling: a CDI word is assigned every ImageNet class whose WordNet
synset is one of the word's anchor synsets or a hyponym of one of them
(transitively, via hypernym + instance-hypernym links).

ANCHORS picks the sense(s) of each CDI word -- the WordNet first sense is
often wrong for child vocabulary (e.g. "arm" -> weapon, "toy" -> toy dog,
"wind" -> wind instrument), so every mapped word is listed explicitly.
Words absent from ANCHORS have no ImageNet referent (body parts, rooms,
substances, and objects ImageNet does not cover, e.g. giraffe, cake, spoon-
like items it lacks).

Where WordNet's taxonomy splits what a child would call one thing (a coffee
mug is not a hyponym of cup.n.01; locomotives are not hyponyms of
train.n.01), extra anchors are added and marked in NOTES.

EXCLUDE removes specific pooled ImageNet classes (by synset name) that are
technically hyponyms but are not what the CDI word refers to.

In [4]:
from nltk.corpus import wordnet as wn

IMAGENET_ANCHORS = {
    "airplane": ["airplane.n.01", "warplane.n.01"],
    "alligator": ["alligator.n.02"],
    "animal": ["animal.n.01"],
    "ant": ["ant.n.01"],
    "apple": ["apple.n.01"],
    "ball": ["ball.n.01"],
    "balloon": ["balloon.n.01"],
    "banana": ["banana.n.02"],
    "basket": ["basket.n.01"],
    "bathtub": ["bathtub.n.01"],
    "bear": ["bear.n.01"],
    "bed": ["bed.n.01"],
    "bee": ["bee.n.01"],
    "bench": ["bench.n.01"],
    "bib": ["bib.n.01"],
    "bicycle": ["bicycle.n.01"],
    "bird": ["bird.n.01"],
    "blanket": ["blanket.n.01", "quilt.n.01"],
    "boat": ["boat.n.01"],
    "boots": ["boot.n.01"],
    "bottle": ["bottle.n.01"],
    "bowl": ["bowl.n.03"],
    "box": ["box.n.01"],
    "bread": ["bread.n.01"],
    "broom": ["broom.n.01"],
    "brush": ["brush.n.02"],
    "bubbles": ["bubble.n.01"],
    "bucket": ["bucket.n.01"],
    "bug": ["insect.n.01"],
    "bunny": ["rabbit.n.01"],
    "bus": ["bus.n.01"],
    "butterfly": ["butterfly.n.01"],
    "camera": ["camera.n.01"],
    "can": ["can.n.01"],
    "car": ["car.n.01"],
    "cat": ["domestic_cat.n.01"],
    "chair": ["chair.n.01"],
    "chicken": ["cock.n.05", "hen.n.02"],
    "clock": ["clock.n.01"],
    "closet": ["wardrobe.n.01"],
    "coat": ["coat.n.01"],
    "coffee": ["coffee.n.01"],
    "corn": ["corn.n.02"],
    "couch": ["sofa.n.01"],
    "cow": ["cattle.n.01"],
    "cracker": ["cracker.n.01"],
    "crib": ["crib.n.01"],
    "cup": ["cup.n.01", "cup.n.06", "mug.n.04"],
    "diaper": ["diaper.n.01"],
    "dish": ["dish.n.01"],
    "dog": ["dog.n.01"],
    "door": ["door.n.01"],
    "dress": ["dress.n.01"],
    "drink": ["beverage.n.01"],
    "duck": ["duck.n.01"],
    "elephant": ["elephant.n.01"],
    "firetruck": ["fire_engine.n.01"],
    "fish": ["fish.n.01"],
    "flower": ["flower.n.01"],
    "food": ["food.n.01", "food.n.02"],
    "frog": ["frog.n.01"],
    "glass": ["glass.n.02"],
    "glasses": ["spectacles.n.01"],
    "glove": ["glove.n.02"],
    "goose": ["goose.n.01"],
    "hamburger": ["hamburger.n.01"],
    "hammer": ["hammer.n.02"],
    "hat": ["hat.n.01"],
    "hen": ["hen.n.02"],
    "horse": ["horse.n.01"],
    "jeans": ["jean.n.01"],
    "knife": ["knife.n.01"],
    "lamp": ["lamp.n.02"],
    "light": ["lamp.n.01", "lamp.n.02", "light.n.02"],
    "lion": ["lion.n.01"],
    "mitten": ["mitten.n.01"],
    "monkey": ["monkey.n.01"],
    "mop": ["swab.n.02"],
    "motorcycle": ["motorcycle.n.01"],
    "nail": ["nail.n.02"],
    "necklace": ["necklace.n.01"],
    "orange": ["orange.n.01"],
    "oven": ["oven.n.01", "microwave.n.02"],
    "owl": ["owl.n.01"],
    "pajamas": ["pajama.n.02"],
    "pants": ["trouser.n.01"],
    "paper": ["paper.n.01"],
    "pen": ["pen.n.01"],
    "penguin": ["penguin.n.01"],
    "person": ["person.n.01"],
    "pig": ["hog.n.03"],
    "pillow": ["pillow.n.01"],
    "pizza": ["pizza.n.01"],
    "plant": ["plant.n.02"],
    "plate": ["plate.n.08"],
    "popsicle": ["ice_lolly.n.01"],
    "potato": ["potato.n.01"],
    "pretzel": ["pretzel.n.01"],
    "purse": ["purse.n.03"],
    "puzzle": ["puzzle.n.02"],
    "radio": ["radio.n.03"],
    "refrigerator": ["refrigerator.n.01"],
    "roof": ["roof.n.01"],
    "rooster": ["cock.n.05"],
    "sandwich": ["sandwich.n.01"],
    "sauce": ["sauce.n.01"],
    "scarf": ["scarf.n.01"],
    "sheep": ["sheep.n.01"],
    "shirt": ["shirt.n.01"],
    "shoe": ["shoe.n.01"],
    "shovel": ["shovel.n.01"],
    "sink": ["sink.n.01", "washbasin.n.02"],
    "sled": ["sled.n.01"],
    "sneaker": ["gym_shoe.n.01", "running_shoe.n.01"],
    "sock": ["sock.n.01"],
    "soup": ["soup.n.01"],
    "spoon": ["spoon.n.01"],
    "squirrel": ["squirrel.n.01"],
    "stove": ["stove.n.02"],
    "strawberry": ["strawberry.n.01"],
    "sweater": ["sweater.n.01"],
    "swing": ["swing.n.02"],
    "table": ["table.n.02", "table.n.03"],
    "teddybear": ["teddy.n.01"],
    "telephone": ["telephone.n.01"],
    "tiger": ["tiger.n.02"],
    "tissue": ["tissue.n.02"],
    "towel": ["towel.n.01"],
    "toy": ["plaything.n.01"],
    "tractor": ["tractor.n.01"],
    "train": ["train.n.01", "locomotive.n.01", "car.n.02"],
    "tray": ["tray.n.01"],
    "tricycle": ["tricycle.n.01"],
    "truck": ["truck.n.01"],
    "turtle": ["turtle.n.02"],
    "tv": ["television.n.02", "television_receiver.n.01"],
    "vacuum": ["vacuum.n.04"],
    "watch": ["watch.n.01"],
    "wolf": ["wolf.n.01"],
    "zebra": ["zebra.n.01"],
}

IMAGENET_EXCLUDE = {
    "ball": {"punching_bag.n.02": "punching bag, not a ball"},
    "bread": {"meat_loaf.n.01": "WordNet files meat loaf under loaf/bread"},
    "dish": {"petri_dish.n.01": "lab glassware"},
    "drink": {"cup.n.06": "ImageNet 'cup' images show the container (mapped to cup)"},
    "food": {
        "menu.n.02": "a menu, not food",
        "hay.n.01": "animal fodder",
        "cup.n.06": "ImageNet 'cup' images show the container (mapped to cup)",
    },
    "oven": {"dutch_oven.n.01": "a cooking pot"},
    "truck": {"minivan.n.01": "passenger vehicle; kept under car"},
}

IMAGENET_NOTES = {
    "airplane": "warplane added: WordNet files military planes outside airplane.n.01",
    "blanket": "quilt/comforter added: WordNet files it under bedclothes, not blanket",
    "cat": "domestic_cat.n.01 rather than cat.n.01 (which pools cougar, lynx)",
    "chicken": "CDI domain is food_drink, but no cooked-chicken ImageNet class; mapped to the bird classes (cock, hen)",
    "cup": "ImageNet 'cup' is cup.n.06 (filed under punch); coffee mug added",
    "fish": "CDI domain is food_drink; mapped to fish.n.01 (the animal), the visual referent",
    "light": "light sources (candle, jack-o'-lantern, spotlight, torch) plus table lamp",
    "oven": "microwave added: WordNet files it under kitchen_appliance, not oven",
    "pig": "hog.n.03 (domestic pig) rather than swine.n.01 (pools wild boar, warthog)",
    "plate": "ImageNet 'plate' is plate.n.08 (a plated dish of food)",
    "sink": "washbasin added: WordNet files it under basin, not sink",
    "sneaker": "running_shoe added: WordNet files it under shoe, not gym_shoe",
    "train": "locomotives and rail cars added: not hyponyms of train.n.01",
}

## LVIS anchors
WordNet anchors for mapping CDI nouns onto LVIS v1 categories (hyponym pooling).

Starts from the ImageNet anchors (cdi_imagenet_anchors.py) so a CDI word means
the same thing in both datasets, then applies LVIS-specific changes. Kept
separate so the ImageNet mapping (and every analysis built on it) is unchanged.

  LVIS_ADD      extra anchors: CDI words ImageNet had no class for, or LVIS
                filing the same object under a different synset (e.g. LVIS
                "plate" is plate.n.04, ImageNet's is plate.n.08)
  LVIS_REPLACE  anchors replaced outright for LVIS
  LVIS_EXCLUDE  pooled LVIS categories that are not the CDI referent

In [5]:
LVIS_ADD = {
    # valid129 words with a direct LVIS category
    "belt": ["belt.n.02"],
    "bib": ["bib.n.02"],
    "book": ["book.n.01"],
    "carrot": ["carrot.n.01"],
    "chips": ["chip.n.04"],
    "cookie": ["cookie.n.01"],
    "doll": ["doll.n.01"],
    "drawer": ["drawer.n.01"],
    "egg": ["egg.n.02"],
    "fork": ["fork.n.01"],
    "hose": ["hose.n.03"],
    "jacket": ["jacket.n.01"],
    "jar": ["jar.n.01"],
    "ladder": ["ladder.n.01"],
    "melon": ["melon.n.01"],
    "milk": ["milk.n.01"],
    "muffin": ["muffin.n.01"],
    "napkin": ["napkin.n.01"],
    "pickle": ["pickle.n.01"],
    "pony": ["pony.n.05"],
    "pumpkin": ["pumpkin.n.02"],
    "scissors": ["scissors.n.01"],
    "shorts": ["short_pants.n.01"],
    "shower": ["shower.n.01"],
    "slide": ["chute.n.02"],
    "slipper": ["slipper.n.01"],
    "soap": ["soap.n.01"],
    "stroller": ["baby_buggy.n.01"],
    "tape": ["tape.n.01"],
    "toothbrush": ["toothbrush.n.01"],
    # same object, different synset in LVIS
    "chicken": ["chicken.n.02"],
    "closet": ["cupboard.n.01"],
    "corn": ["corn.n.03"],
    "flower": ["flower_arrangement.n.01", "bouquet.n.01"],
    "glove": ["baseball_glove.n.01"],
    "plate": ["plate.n.04"],
    "purse": ["bag.n.04"],
    "radio": ["radio_receiver.n.01"],
    "rooster": ["cock.n.04"],
    "stove": ["stove.n.01"],
    "toy": ["toy.n.03"],
    # non-valid129 words with a direct LVIS category
    "beads": ["bead.n.01"],
    "butter": ["butter.n.01"],
    "button": ["button.n.01"],
    "cake": ["cake.n.03"],
    "crayon": ["crayon.n.01"],
    "deer": ["deer.n.01"],
    "donkey": ["domestic_ass.n.01"],
    "flag": ["flag.n.01"],
    "giraffe": ["giraffe.n.01"],
    "grapes": ["grape.n.01"],
    "helicopter": ["helicopter.n.01"],
    "key": ["key.n.01"],
    "kitty": ["kitten.n.01"],
    "lamb": ["lamb.n.01"],
    "lollipop": ["lollipop.n.02"],
    "money": ["money.n.03"],
    "pancake": ["pancake.n.01"],
    "pastry": ["pastry.n.02"],
    "peas": ["pea.n.01"],
    "pencil": ["pencil.n.01"],
    "puppy": ["puppy.n.01"],
    "snowman": ["snowman.n.01"],
    "soda": ["pop.n.02"],
    "tights": ["tights.n.01"],
    "toast": ["toast.n.01"],
    "yogurt": ["yogurt.n.01"],
}

LVIS_REPLACE = {
    "cat": ["cat.n.01"],      # LVIS "cat" is cat.n.01; cougar excluded below
    "plant": ["flower.n.01"],  # plant.n.02 pools broccoli, lettuce, celery... in LVIS
}

LVIS_EXCLUDE = {
    "bed": {"hammock.n.02": "hammock"},
    "cat": {"cougar.n.01": "wild cat"},
    "box": {"cash_register.n.01": "cash register", "window_box.n.01": "planter"},
    "jar": {"vase.n.01": "vase", "crock.n.03": "crock pot (cooker)", "urn.n.01": "urn"},
    "nail": {"thumbtack.n.01": "thumbtack, not a nail"},
    "refrigerator": {"cooler.n.01": "portable cooler"},
    "toy": {"kite.n.03": "kite (13k sky kites would dominate)", "frisbee.n.01": "sports frisbee"},
}

LVIS_NOTES = {
    "cat": "LVIS cat is cat.n.01 (ImageNet anchor domestic_cat.n.01 is below it)",
    "plant": "anchored on flower.n.01 as for ImageNet; plant.n.02 pools vegetables in LVIS",
    "toy": "plaything.n.01 + LVIS generic toy (toy.n.03), minus kite/frisbee",
    "flower": "flower arrangements and bouquets added",
    "glove": "baseball glove added: not a hyponym of glove.n.02",
}

LVIS_ANCHORS = {w: list(a) for w, a in IMAGENET_ANCHORS.items()}

for w, extra in LVIS_ADD.items():
    LVIS_ANCHORS[w] = LVIS_ANCHORS.get(w, []) + [a for a in extra if a not in LVIS_ANCHORS.get(w, [])]

LVIS_ANCHORS.update(LVIS_REPLACE)

LVIS_EXCLUDE_ALL = {w: dict(d) for w, d in IMAGENET_EXCLUDE.items()}

for w, d in LVIS_EXCLUDE.items():
    LVIS_EXCLUDE_ALL.setdefault(w, {}).update(d)

Save both anchor sets for the mapping audit (notebook 02).

In [6]:
MAPPING_OUT.mkdir(parents=True, exist_ok=True)
for ds, (a, e, n) in {"imagenet": (IMAGENET_ANCHORS, IMAGENET_EXCLUDE, IMAGENET_NOTES),
                      "lvis": (LVIS_ANCHORS, LVIS_EXCLUDE_ALL, LVIS_NOTES)}.items():
    (MAPPING_OUT / f"anchors_{ds}.json").write_text(json.dumps({"anchors": a, "exclude": e, "notes": n}, indent=1))

## Shared mapping helpers
`hypernym_distances` measures how many hypernym steps a class is from an anchor.

In [7]:
def count_images(split_dir: Path) -> Counter:
    """Images per wnid; filenames end in _<wnid>.JPEG (see unpack_imagenet_lvis.sh)."""
    counts = Counter()
    with os.scandir(split_dir) as it:
        for e in it:
            if e.name.endswith(".JPEG"):
                counts[e.name[:-5].rsplit("_", 1)[1]] += 1
    return counts


def hypernym_distances(s):
    """{ancestor synset: shortest hypernym-path distance}, including s itself at 0."""
    dist = {s: 0}
    frontier = [s]
    while frontier:
        nxt = []
        for x in frontier:
            for h in x.hypernyms() + x.instance_hypernyms():
                if h not in dist:
                    dist[h] = dist[x] + 1
                    nxt.append(h)
        frontier = nxt
    return dist

## ImageNet mapping

In [8]:
def map_imagenet(imagenet_dir: Path = IMAGENET_DIR, out_dir: Path = MAPPING_OUT):
    """Build the CDI <-> ImageNet map; write and return the pair, summary and per-class tables."""
    out_dir.mkdir(parents=True, exist_ok=True)

    train_n = count_images(imagenet_dir / "train")
    val_n = count_images(imagenet_dir / "val")
    wnids = sorted(val_n)
    assert len(wnids) == 1000, f"expected 1000 ImageNet classes, found {len(wnids)}"
    syn = {w: wn.synset_from_pos_and_offset("n", int(w[1:])) for w in wnids}
    dists = {w: hypernym_distances(s) for w, s in syn.items()}

    cdi = pd.read_csv(VOCAB_DIR / "cdi_concrete_nouns_295_with_semantic.csv")
    valid129 = set((VOCAB_DIR / "included_categories_valid129.txt").read_text().split())
    valid85 = set((VOCAB_DIR / "included_categories_valid85.txt").read_text().split())
    unknown = set(IMAGENET_ANCHORS) - set(cdi["category"])
    assert not unknown, f"IMAGENET_ANCHORS keys not in CDI list: {unknown}"

    rows = []
    for word, anchor_names in IMAGENET_ANCHORS.items():
        anchors = [wn.synset(a) for a in anchor_names]
        excl = IMAGENET_EXCLUDE.get(word, {})
        unused = set(excl) - {syn[w].name() for w in wnids}
        assert not unused, f"{word}: IMAGENET_EXCLUDE names not ImageNet classes: {unused}"
        for w in wnids:
            if syn[w].name() in excl:
                continue
            hits = [(dists[w][a], a) for a in anchors if a in dists[w]]
            if not hits:
                continue
            d, a = min(hits, key=lambda t: t[0])
            rows.append({
                "cdi_category": word,
                "wnid": w,
                "imagenet_synset": syn[w].name(),
                "imagenet_label": ", ".join(l.replace("_", " ") for l in syn[w].lemma_names()),
                "anchor_synset": a.name(),
                "hypernym_distance": d,
                "n_train": train_n[w],
                "n_val": val_n[w],
            })
    pairs = pd.DataFrame(rows)

    # most_specific: closest CDI anchor(s) for each ImageNet class (ties kept).
    min_d = pairs.groupby("wnid")["hypernym_distance"].transform("min")
    pairs["most_specific"] = pairs["hypernym_distance"] == min_d
    pairs = pairs.merge(cdi.rename(columns={"category": "cdi_category"}), on="cdi_category")
    pairs = pairs.sort_values(["cdi_category", "hypernym_distance", "wnid"])
    pairs.to_csv(out_dir / "imagenet_cdi_hyponym_map.csv", index=False)

    g = pairs.groupby("cdi_category")
    summary = cdi.rename(columns={"category": "cdi_category"}).assign(
        in_valid129=lambda d: d["cdi_category"].isin(valid129),
        in_valid85=lambda d: d["cdi_category"].isin(valid85),
        anchor_synsets=lambda d: d["cdi_category"].map(lambda c: ";".join(IMAGENET_ANCHORS.get(c, []))),
        n_imagenet_classes=lambda d: d["cdi_category"].map(g.size()).fillna(0).astype(int),
        n_classes_most_specific=lambda d: d["cdi_category"].map(g["most_specific"].sum()).fillna(0).astype(int),
        n_train_images=lambda d: d["cdi_category"].map(g["n_train"].sum()).fillna(0).astype(int),
        n_val_images=lambda d: d["cdi_category"].map(g["n_val"].sum()).fillna(0).astype(int),
        imagenet_classes=lambda d: d["cdi_category"].map(
            g["imagenet_synset"].agg(lambda s: ";".join(x.split(".")[0] for x in s))).fillna(""),
        excluded=lambda d: d["cdi_category"].map(lambda c: ";".join(IMAGENET_EXCLUDE.get(c, {}))),
        notes=lambda d: d["cdi_category"].map(lambda c: IMAGENET_NOTES.get(c, "")),
    )
    summary.to_csv(out_dir / "cdi_imagenet_summary.csv", index=False)

    spec = pairs[pairs["most_specific"]].groupby("wnid").agg(
        cdi_most_specific=("cdi_category", lambda s: "|".join(sorted(s))))
    allc = pairs.groupby("wnid").agg(cdi_all=("cdi_category", lambda s: "|".join(sorted(s))))
    per_class = pd.DataFrame({
        "wnid": wnids,
        "imagenet_synset": [syn[w].name() for w in wnids],
        "imagenet_label": [", ".join(l.replace("_", " ") for l in syn[w].lemma_names()) for w in wnids],
    }).join(spec, on="wnid").join(allc, on="wnid").fillna({"cdi_most_specific": "", "cdi_all": ""})
    per_class.to_csv(out_dir / "imagenet_class_to_cdi.csv", index=False)

    mapped = summary[summary["n_imagenet_classes"] > 0]
    print(f"CDI words mapped: {len(mapped)}/295 "
          f"(valid129: {mapped['in_valid129'].sum()}/129, valid85: {mapped['in_valid85'].sum()}/85)")
    print(f"ImageNet classes with >=1 CDI word: {(per_class['cdi_all'] != '').sum()}/1000; "
          f"excluding 'animal'/'food'/'bug': "
          f"{pairs[~pairs['cdi_category'].isin(['animal', 'food', 'bug'])]['wnid'].nunique()}")
    print(f"wrote {out_dir}")

    return {"pairs": pairs, "summary": summary, "per_class": per_class}

## LVIS mapping and crop manifest

In [9]:
def map_lvis(lvis_root: Path = LVIS_ROOT, out_dir: Path = MAPPING_OUT):
    """Build the CDI <-> LVIS map and the crop manifest; write and return the pair, summary and manifest tables."""
    out_dir.mkdir(parents=True, exist_ok=True)

    splits = {s: json.load(open(lvis_root / "annotations" / f"lvis_v1_{s}.json")) for s in ("train", "val")}
    cats = splits["val"]["categories"]
    ann = pd.concat([
        pd.DataFrame(d["annotations"])[["id", "image_id", "category_id", "bbox", "area"]].assign(lvis_split=s)
        for s, d in splits.items()
    ], ignore_index=True)
    # LVIS v1 images come from both COCO train2017 and val2017; coco_url says which.
    img = pd.concat([pd.DataFrame(d["images"])[["id", "coco_url", "width", "height"]] for d in splits.values()])
    img = img.drop_duplicates("id").set_index("id")
    n_inst = ann.groupby(["category_id", "lvis_split"]).size().unstack(fill_value=0)

    syn, dists = {}, {}
    for c in cats:
        try:
            syn[c["id"]] = wn.synset(c["synset"])
        except Exception:
            print(f"skip LVIS category {c['name']}: synset {c['synset']} not in NLTK WordNet")
            continue
        dists[c["id"]] = hypernym_distances(syn[c["id"]])
    name = {c["id"]: c["name"] for c in cats}

    cdi = pd.read_csv(VOCAB_DIR / "cdi_concrete_nouns_295_with_semantic.csv")
    valid129 = set((VOCAB_DIR / "included_categories_valid129.txt").read_text().split())
    valid85 = set((VOCAB_DIR / "included_categories_valid85.txt").read_text().split())
    assert not set(LVIS_ANCHORS) - set(cdi["category"]), set(LVIS_ANCHORS) - set(cdi["category"])

    rows = []
    for word, anchor_names in LVIS_ANCHORS.items():
        anchors = [wn.synset(a) for a in anchor_names]
        excl = LVIS_EXCLUDE_ALL.get(word, {})
        for cid, d in dists.items():
            if syn[cid].name() in excl:
                continue
            hits = [(d[a], a) for a in anchors if a in d]
            if not hits:
                continue
            dist, a = min(hits, key=lambda t: t[0])
            rows.append({
                "cdi_category": word, "lvis_category_id": cid, "lvis_name": name[cid],
                "lvis_synset": syn[cid].name(), "anchor_synset": a.name(), "hypernym_distance": dist,
                "n_train": int(n_inst.loc[cid, "train"]) if cid in n_inst.index else 0,
                "n_val": int(n_inst.loc[cid, "val"]) if cid in n_inst.index else 0,
            })
    pairs = pd.DataFrame(rows)
    min_d = pairs.groupby("lvis_category_id")["hypernym_distance"].transform("min")
    pairs["most_specific"] = pairs["hypernym_distance"] == min_d
    pairs = pairs.merge(cdi.rename(columns={"category": "cdi_category"}), on="cdi_category")
    pairs = pairs.sort_values(["cdi_category", "hypernym_distance", "lvis_name"])
    pairs.to_csv(out_dir / "lvis_cdi_hyponym_map.csv", index=False)

    g = pairs.assign(n=pairs["n_train"] + pairs["n_val"]).groupby("cdi_category")
    summary = cdi.rename(columns={"category": "cdi_category"}).assign(
        in_valid129=lambda d: d["cdi_category"].isin(valid129),
        in_valid85=lambda d: d["cdi_category"].isin(valid85),
        anchor_synsets=lambda d: d["cdi_category"].map(lambda c: ";".join(LVIS_ANCHORS.get(c, []))),
        n_lvis_categories=lambda d: d["cdi_category"].map(g.size()).fillna(0).astype(int),
        n_instances=lambda d: d["cdi_category"].map(g["n"].sum()).fillna(0).astype(int),
        lvis_categories=lambda d: d["cdi_category"].map(g["lvis_name"].agg(";".join)).fillna(""),
        excluded=lambda d: d["cdi_category"].map(lambda c: ";".join(LVIS_EXCLUDE_ALL.get(c, {}))),
        notes=lambda d: d["cdi_category"].map(lambda c: LVIS_NOTES.get(c, "")),
    )
    summary.to_csv(out_dir / "cdi_lvis_summary.csv", index=False)

    # Crop manifest: every box of every mapped LVIS category.
    keep = ann[ann["category_id"].isin(set(pairs["lvis_category_id"]))].copy()
    xywh = pd.DataFrame(keep["bbox"].tolist(), index=keep.index, columns=["x", "y", "w", "h"])
    keep = keep[(xywh["w"] > 0) & (xywh["h"] > 0)]
    xywh = xywh.loc[keep.index]
    url = img.loc[keep["image_id"], "coco_url"].to_numpy()
    folder = [u.rsplit("/", 2)[1] for u in url]  # train2017 | val2017
    fname = [u.rsplit("/", 1)[1] for u in url]
    # LVIS annotation ids restart at 1 in each split, so the split is part of the id.
    keep["image_id_out"] = "lvis_" + keep["lvis_split"] + "_" + keep["id"].astype(str)
    keep["path"] = [str(lvis_root / "extracted" / f / n) for f, n in zip(folder, fname)]
    manifest = pd.DataFrame({
        "image_id": keep["image_id_out"], "path": keep["path"],
        "xmin": xywh["x"], "ymin": xywh["y"], "xmax": xywh["x"] + xywh["w"], "ymax": xywh["y"] + xywh["h"],
    })
    manifest.to_csv(out_dir / "lvis_crop_manifest.csv", index=False)
    keep.assign(lvis_name=keep["category_id"].map(name), coco_folder=folder)[
        ["image_id_out", "id", "image_id", "category_id", "lvis_name", "lvis_split", "coco_folder", "area"]
    ].rename(columns={"image_id_out": "crop_id", "id": "lvis_ann_id", "image_id": "coco_image_id"}).to_csv(
        out_dir / "lvis_crop_metadata.csv.gz", index=False)

    mapped = summary[summary["n_lvis_categories"] > 0]
    v = mapped[mapped["in_valid129"]]
    print(f"CDI words mapped: {len(mapped)}/295 (valid129: {len(v)}/129, of which >=100 instances: "
          f"{(v['n_instances'] >= 100).sum()}; valid85: {mapped['in_valid85'].sum()}/85)")
    print(f"LVIS categories mapped: {pairs['lvis_category_id'].nunique()}/{len(cats)}; "
          f"crops in manifest: {len(manifest)} (dropped zero-area: {len(ann[ann['category_id'].isin(set(pairs['lvis_category_id']))]) - len(keep)})")
    print(f"crops for valid129-mapped LVIS categories: "
          f"{keep['category_id'].isin(set(pairs.loc[pairs['cdi_category'].isin(valid129), 'lvis_category_id'])).sum()}")

    return {"pairs": pairs, "summary": summary, "manifest": manifest}

## Run (~1 min either way)

In [10]:
res = map_imagenet() if DATASET == "imagenet" else map_lvis()

CDI words mapped: 140/295 (valid129: 79/129, valid85: 56/85)
ImageNet classes with >=1 CDI word: 611/1000; excluding 'animal'/'food'/'bug': 435
wrote /home/j7yang/babyview-projects/vss2026/object-detection/analysis/comparison-datasets/outputs/mapping


## Coverage of the valid129 categories

In [11]:
s = res["summary"]
n_col = "n_imagenet_classes" if DATASET == "imagenet" else "n_lvis_categories"
v = s[s.in_valid129]
print(f"valid129 words with >=1 pooled class: {(v[n_col] > 0).sum()} / 129")
print("no class in this dataset:", ", ".join(v.loc[v[n_col] == 0, "cdi_category"]))
v.sort_values(n_col, ascending=False)[["cdi_category", "cdi_semantic", "anchor_synsets", n_col,
                                       "imagenet_classes" if DATASET == "imagenet" else "lvis_categories", "excluded"]].head(25)

valid129 words with >=1 pooled class: 79 / 129
no class in this dataset: ankle, arm, belt, book, carrot, chips, comb, cookie, doll, drawer, ear, egg, eye, face, finger, foot, fork, grass, gum, hair, hand, hose, jacket, jar, ladder, melon, milk, mouth, muffin, napkin, nose, pickle, pony, pumpkin, rock, scissors, shorts, shower, slide, slipper, soap, sprinkler, stick, stroller, tape, toe, tooth, toothbrush, tree, window


,cdi_category,cdi_semantic,anchor_synsets,n_imagenet_classes,imagenet_classes,excluded
82,dog,animals,dog.n.01,118,dalmatian;basenji;pug;leonberg;newfoundland;great_pyrenees;mexican_hairless;chihuahua;japanese_spaniel;maltese_dog;p...,
26,bird,animals,bird.n.01,59,cock;hen;ostrich;vulture;african_grey;macaw;bee_eater;hornbill;hummingbird;jacamar;toucan;water_ouzel;kite;bald_eagl...,
98,fish,food_drink,fish.n.01,16,barracouta;coho;hammerhead;electric_ray;stingray;eel;sturgeon;gar;great_white_shark;tiger_shark;puffer;tench;goldfis...,
51,car,vehicles,car.n.01,10,ambulance;beach_wagon;cab;convertible;jeep;limousine;minivan;model_t;racer;sports_car,
8,ball,toys,ball.n.01,9,baseball;basketball;croquet_ball;golf_ball;ping-pong_ball;rugby_ball;soccer_ball;tennis_ball;volleyball,punching_bag.n.02
32,bottle,household,bottle.n.01,7,beer_bottle;pill_bottle;pop_bottle;water_bottle;wine_bottle;water_jug;whiskey_jug,
34,box,household,box.n.01,6,carton;chest;crate;mailbox;pencil_box;safe,
45,butterfly,animals,butterfly.n.01,6,ringlet;sulphur_butterfly;lycaenid;admiral;monarch;cabbage_butterfly,
53,cat,animals,domestic_cat.n.01,5,tabby;tiger_cat;persian_cat;siamese_cat;egyptian_cat,
55,chair,furniture_rooms,chair.n.01,4,barber_chair;folding_chair;rocking_chair;throne,


## Inspect one word's pool
`hypernym_distance` = steps from the anchor; `most_specific` = this word is the closest CDI word for the class.

In [12]:
WORD = "toy"
p = res["pairs"]
p[p.cdi_category == WORD].drop(columns=[c for c in ["cdi_semantic", "uni_lemma"] if c in p.columns])

,cdi_category,wnid,imagenet_synset,imagenet_label,anchor_synset,hypernym_distance,n_train,n_val,most_specific
900,toy,n04371774,swing.n.02,swing,plaything.n.01,1,1300,50,False
901,toy,n04399382,teddy.n.01,"teddy, teddy bear",plaything.n.01,1,1300,50,False


### Review checklist

- Do the anchor senses match what a child means by each word?
- Do the large pools (dog, bird, toy) contain only things the CDI word refers to?
- Are the exclusions justified; should other hyponyms be excluded?
- LVIS: does the crop manifest drop only zero-area boxes?